# 10. Is pressure judged per frame or in the context of the sequence?

Observation (quick check): at the same nearest-defender distance, a pass is more often labelled 'pressured' (P)
when the previous pass in the same team sequence was labelled P.

Three checks:
1. **Confounding** — does the previous label still matter after controlling for the current situation
   (distance, second defender, defenders within 3/5 m, approach speed, pitch location)?
2. **Label or situation?** — is it the previous *human label* or the previous *tracking distance* that carries over?
3. **Time decay** — does the carry-over fade with the time between the two passes?

Plus the same question **within one player's possession**: does the reception label (`initialPressureType`)
carry over to the pass label, beyond the tracking distances at reception and at the pass?

All 64 matches are used. P vs N passes only as the outcome; the previous pass can have any label.

In [1]:
import os, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, SplineTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

from football.common import KEY, cv_predict, boot_diff

OUT = os.path.join("..", "outputs")
FIG = os.path.join(OUT, "figs")
DATA = os.path.join("..", "data", "World_cup", "Event Data")
ev = pd.read_csv(os.path.join(OUT, "events.csv.gz"), low_memory=False)
E = pd.read_csv(os.path.join(OUT, "pressure_events.csv.gz"))
W = pd.read_csv(os.path.join(OUT, "possession_distances.csv.gz"))[KEY + ["d_start", "d_min", "n_frames"]]
print(len(E), len(W))

61012 60483


## 1. Reception labels (raw event files) and sequence order

In [2]:
CACHE = os.path.join(OUT, "reception_labels.csv.gz")
if not os.path.exists(CACHE):
    rows = []
    for f in sorted(glob.glob(os.path.join(DATA, "*_event.json"))):
        for e in json.load(open(f, encoding="utf-8")):
            pe = e.get("possessionEvents") or {}
            if pe.get("possessionEventType") == "PA":
                rows.append(dict(game_id=e["gameId"], pe_id=e["possessionEventId"],
                                 init_pressure=(e.get("initialTouch") or {}).get("initialPressureType")))
    pd.DataFrame(rows).to_csv(CACHE, index=False)
R = pd.read_csv(CACHE)

pa = ev[ev.event_type == "PA"][KEY + ["time", "period", "team_id", "sequence", "actor_id"]]
pa = pa.merge(E, on=KEY, how="left").merge(R, on=KEY, how="left").merge(W, on=KEY, how="left")
pa = pa.sort_values(["game_id", "period", "time"]).reset_index(drop=True)
grp = pa.groupby(["game_id", "period", "sequence", "team_id"], sort=False)
pa["prev_label"] = grp.label.shift(1)
pa["prev_d1"] = grp.d1.shift(1)
pa["gap"] = pa.time - grp.time.shift(1)
pa["abs_y"] = pa.cy.abs()
X = pa[pa.label.isin(["P", "N"]) & pa.prev_label.isin(["P", "N", "A", "L"]) & (pa.gap <= 10)].copy()
X["y"] = (X.label == "P").astype(int)
for lab in "PAL":
    X[f"prev_{lab}"] = (X.prev_label == lab).astype(int)
print(f"passes with a previous pass in the same sequence within 10 s: {len(X):,}  (P share {X.y.mean():.3f})")
X.groupby("prev_label").y.agg(["mean", "size"]).round(3)

passes with a previous pass in the same sequence within 10 s: 38,984  (P share 0.396)


,mean,size
prev_label,,
A,0.374,2295
L,0.413,2483
N,0.334,21288
P,0.500,12918


## 2. Does the previous label matter beyond the current situation? (nested models, game-grouped CV)

In [3]:
CUR = ["d1", "d2", "n3", "n5", "closing_max", "cx", "abs_y"]
CONT = {"d1", "d2", "closing_max", "cx", "abs_y", "prev_d1", "gap"}
X["vel_ok"] = X.vel_ok_pass.astype(int)


def mk(cols):
    ct = ColumnTransformer([(c, SplineTransformer(n_knots=4), [c]) if c in CONT else (c, "passthrough", [c]) for c in cols])
    return lambda s: make_pipeline(ct, StandardScaler(), LogisticRegression(max_iter=3000))


MODELS = {
    "A current situation": CUR + ["vel_ok"],
    "B + previous distance (tracking)": CUR + ["vel_ok", "prev_d1"],
    "C + previous label (human)": CUR + ["vel_ok", "prev_d1", "prev_P", "prev_A", "prev_L"],
}
y, g = X.y, X.game_id
Xf = X[sorted(set(sum(MODELS.values(), [])))].astype(float).fillna(X.prev_d1.median())
preds, rows, prev = {}, [], None
for name, cols in MODELS.items():
    preds[name] = cv_predict(mk(cols), Xf[cols], y, g, seeds=(0,))
    row = dict(model=name, roc_auc=roc_auc_score(y, preds[name]))
    if prev:
        d, ci, _ = boot_diff(y, preds[name], preds[prev], roc_auc_score, n=300, groups=g)
        row.update(gain=d, ci95=f"[{ci[0]:+.4f}, {ci[1]:+.4f}]")
    rows.append(row); prev = name
nested = pd.DataFrame(rows).round(4)
nested

,model,roc_auc,gain,ci95
0,A current situation,0.9057,NaN,NaN
1,B + previous distance (tracking),0.9059,0.0001,"[-0.0001, +0.0003]"
2,C + previous label (human),0.9082,0.0024,"[+0.0013, +0.0035]"


### Effect size: odds ratio of 'previous pass P' (vs previous N), adjusted for the current situation and previous distance
Logistic regression without penalty on the full data; 95% CI from a game-level cluster bootstrap (200 refits).

In [4]:
cols = MODELS["C + previous label (human)"]


def fit_or(D):
    ct = ColumnTransformer([(c, SplineTransformer(n_knots=4), [c]) if c in CONT else (c, "passthrough", [c]) for c in cols])
    m = make_pipeline(ct, LogisticRegression(max_iter=5000, C=1e6)).fit(D[cols].astype(float).fillna(D.prev_d1.median()), D.y)
    names = m[0].get_feature_names_out()
    coef = dict(zip(names, m[-1].coef_[0]))
    return {k: np.exp(coef[f"{k}__{k}"]) for k in ["prev_P", "prev_A", "prev_L"]}


point = fit_or(X)
rng = np.random.RandomState(0)
games = X.game_id.unique()
by_game = {gg: X[X.game_id == gg] for gg in games}
boot = [fit_or(pd.concat([by_game[gg] for gg in rng.choice(games, len(games))])) for _ in range(200)]
ORs = pd.DataFrame({k: [point[k], *np.percentile([b[k] for b in boot], [2.5, 97.5])] for k in point},
                   index=["odds ratio", "2.5%", "97.5%"]).T.round(2)
ORs

,odds ratio,2.5%,97.5%
prev_P,2.09,1.8,2.41
prev_A,1.15,1.0,1.32
prev_L,1.42,1.2,1.67


## 3. Where does it matter? P-rate by current distance × previous label (raw) and model-adjusted difference

In [5]:
X["dbin"] = pd.cut(X.d1, [0, 2, 3, 4, 6, 100])
raw = X.pivot_table(index="dbin", columns="prev_label", values="y", aggfunc="mean", observed=True)[["N", "P"]].round(3)
full = mk(cols)(0).fit(Xf[cols], y)
adj = []
for b, D in X.groupby("dbin", observed=True):
    Dp, Dn = Xf.loc[D.index, cols].copy(), Xf.loc[D.index, cols].copy()
    Dp[["prev_P", "prev_A", "prev_L"]] = [1, 0, 0]; Dn[["prev_P", "prev_A", "prev_L"]] = [0, 0, 0]
    adj.append(dict(dbin=b, adjusted_P_if_prev_N=full.predict_proba(Dn)[:, 1].mean(), adjusted_P_if_prev_P=full.predict_proba(Dp)[:, 1].mean()))
adj = pd.DataFrame(adj).set_index("dbin").round(3)
adj["difference"] = (adj.adjusted_P_if_prev_P - adj.adjusted_P_if_prev_N).round(3)
pd.concat([raw.add_prefix("raw P rate | prev="), adj], axis=1)

,raw P rate | prev=N,raw P rate | prev=P,adjusted_P_if_prev_N,adjusted_P_if_prev_P,difference
dbin,,,,,
"(0, 2]",0.918,0.928,0.908,0.951,0.043
"(2, 3]",0.755,0.814,0.717,0.832,0.115
"(3, 4]",0.399,0.527,0.366,0.528,0.162
"(4, 6]",0.150,0.234,0.169,0.283,0.114
"(6, 100]",0.058,0.111,0.053,0.100,0.047


## 4. Time decay: does the carry-over fade with the time between passes?

In [6]:
X["gapbin"] = pd.cut(X.gap, [0, 1.5, 3, 5, 10])
dec = []
for b, D in X.groupby("gapbin", observed=True):
    Dp, Dn = Xf.loc[D.index, cols].copy(), Xf.loc[D.index, cols].copy()
    Dp[["prev_P", "prev_A", "prev_L"]] = [1, 0, 0]; Dn[["prev_P", "prev_A", "prev_L"]] = [0, 0, 0]
    # refit within the gap bin to let the effect differ by gap
    m = mk(cols)(0).fit(Xf.loc[D.index, cols], D.y)
    dec.append(dict(gap_s=str(b), n=len(D), adjusted_difference=m.predict_proba(Dp)[:, 1].mean() - m.predict_proba(Dn)[:, 1].mean(),
                    raw_P_prevP=D[D.prev_label == "P"].y.mean(), raw_P_prevN=D[D.prev_label == "N"].y.mean()))
decay = pd.DataFrame(dec).round(3)
decay

,gap_s,n,adjusted_difference,raw_P_prevP,raw_P_prevN
0,"(0.0, 1.5]",6821,0.075,0.567,0.398
1,"(1.5, 3.0]",16184,0.081,0.513,0.344
2,"(3.0, 5.0]",11313,0.097,0.440,0.296
3,"(5.0, 10.0]",4666,0.064,0.453,0.314


## 5. Within one player's possession: reception label → pass label
Controls: tracking distance at the pass (d1 / d_end) and at reception (d_start), plus the current situation.

In [7]:
Y = pa[pa.label.isin(["P", "N"]) & pa.init_pressure.notna() & pa.d_start.notna()].copy()
Y["y"] = (Y.label == "P").astype(int)
for lab in "PAL":
    Y[f"rec_{lab}"] = (Y.init_pressure == lab).astype(int)
Y["abs_y"] = Y.cy.abs(); Y["vel_ok"] = Y.vel_ok_pass.astype(int)
CONT.update({"d_start"})
M2 = {"A current situation + distance at reception": CUR + ["vel_ok", "d_start"],
      "B + reception label (human)": CUR + ["vel_ok", "d_start", "rec_P", "rec_A", "rec_L"]}
Yf = Y[sorted(set(sum(M2.values(), [])))].astype(float)
p = {k: cv_predict(mk(c), Yf[c], Y.y, Y.game_id, seeds=(0,)) for k, c in M2.items()}
d, ci, _ = boot_diff(Y.y, p[list(M2)[1]], p[list(M2)[0]], roc_auc_score, n=300, groups=Y.game_id)
print(f"n = {len(Y):,};  AUC {roc_auc_score(Y.y, p[list(M2)[0]]):.4f} -> {roc_auc_score(Y.y, p[list(M2)[1]]):.4f}  gain {d:+.4f} [{ci[0]:+.4f}, {ci[1]:+.4f}]")
Y["dbin"] = pd.cut(Y.d1, [0, 2, 3, 4, 6, 100])
Y.pivot_table(index="dbin", columns="init_pressure", values="y", aggfunc="mean", observed=True)[["N", "P"]].add_prefix("P rate | reception=").round(3)

n = 52,389;  AUC 0.8962 -> 0.9249  gain +0.0285 [+0.0247, +0.0320]


init_pressure,P rate | reception=N,P rate | reception=P
dbin,,
"(0, 2]",0.885,0.940
"(2, 3]",0.688,0.903
"(3, 4]",0.332,0.847
"(4, 6]",0.120,0.787
"(6, 100]",0.050,0.806


## 5b. Is the pass label a copy of the reception label? Does the previous-pass effect survive the reception label?

In [8]:
print("reception label -> pass label (row share):")
display(pd.crosstab(Y.init_pressure, Y.label, normalize="index").round(3))
Z = X.merge(pa[KEY + ["init_pressure"]], on=KEY, how="left", suffixes=("", "_r")).dropna(subset=["init_pressure"])
for lab in "PAL":
    Z[f"rec_{lab}"] = (Z.init_pressure == lab).astype(int)
base_r = CUR + ["vel_ok", "prev_d1", "rec_P", "rec_A", "rec_L"]
full_r = base_r + ["prev_P", "prev_A", "prev_L"]
Zf = Z[full_r].astype(float).fillna(Z.prev_d1.median())
q0 = cv_predict(mk(base_r), Zf[base_r], Z.y, Z.game_id, seeds=(0,))
q1 = cv_predict(mk(full_r), Zf[full_r], Z.y, Z.game_id, seeds=(0,))
d, ci, _ = boot_diff(Z.y, q1, q0, roc_auc_score, n=300, groups=Z.game_id)
print(f"n = {len(Z):,}: + previous pass label beyond reception label: AUC {roc_auc_score(Z.y, q0):.4f} -> {roc_auc_score(Z.y, q1):.4f}, "
      f"gain {d:+.4f} [{ci[0]:+.4f}, {ci[1]:+.4f}]")
ct = ColumnTransformer([(c, SplineTransformer(n_knots=4), [c]) if c in CONT else (c, "passthrough", [c]) for c in full_r])
m = make_pipeline(ct, LogisticRegression(max_iter=5000, C=1e6)).fit(Zf[full_r], Z.y)
coef = dict(zip(m[0].get_feature_names_out(), m[-1].coef_[0]))
rng = np.random.RandomState(0); gg = Z.game_id.unique(); byg = {k: Zf[Z.game_id == k].assign(y=Z.y[Z.game_id == k]) for k in gg}
bo = []
for _ in range(200):
    B_ = pd.concat([byg[k] for k in rng.choice(gg, len(gg))])
    mb = make_pipeline(ColumnTransformer([(c, SplineTransformer(n_knots=4), [c]) if c in CONT else (c, "passthrough", [c]) for c in full_r]),
                       LogisticRegression(max_iter=5000, C=1e6)).fit(B_[full_r], B_.y)
    bo.append(np.exp(dict(zip(mb[0].get_feature_names_out(), mb[-1].coef_[0]))["prev_P__prev_P"]))
or_rec = pd.Series({"odds ratio previous P (adjusted incl. reception label)": np.exp(coef["prev_P__prev_P"]),
                    "2.5%": np.percentile(bo, 2.5), "97.5%": np.percentile(bo, 97.5)}).round(2)
or_rec

reception label -> pass label (row share):


label,N,P
init_pressure,,
A,0.837,0.163
L,0.371,0.629
N,0.703,0.297
P,0.107,0.893


n = 38,984: + previous pass label beyond reception label: AUC 0.9280 -> 0.9290, gain +0.0010 [+0.0004, +0.0016]


odds ratio previous P (adjusted incl. reception label)    1.76
2.5%                                                      1.54
97.5%                                                     2.09
dtype: float64

## 6. Figure

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
x = np.arange(len(adj))
axes[0].bar(x - 0.2, adj.adjusted_P_if_prev_N, 0.4, color="#9ecae1", label="previous pass: no pressure")
axes[0].bar(x + 0.2, adj.adjusted_P_if_prev_P, 0.4, color="#08306b", label="previous pass: pressured")
axes[0].set_xticks(x, [str(i) for i in adj.index]); axes[0].set_xlabel("nearest defender distance at the pass (m)")
axes[0].set_ylabel("adjusted share labelled 'pressured'"); axes[0].legend(frameon=False, fontsize=7)
axes[0].set_title("Same situation, different context", fontsize=9)
axes[1].plot(range(len(decay)), decay.adjusted_difference, marker="o", color="#08306b")
axes[1].set_xticks(range(len(decay)), decay.gap_s); axes[1].set_xlabel("time since previous pass (s)")
axes[1].set_ylabel("effect of previous 'pressured' label"); axes[1].axhline(0, c="grey", lw=0.8)
axes[1].set_title("Carry-over fades with time?", fontsize=9)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig_sequence_context.png"), dpi=300)

## Save

In [10]:
nested.to_csv(os.path.join(OUT, "sequence_nested.csv"), index=False)
ORs.to_csv(os.path.join(OUT, "sequence_odds_ratios.csv"))
adj.to_csv(os.path.join(OUT, "sequence_adjusted_by_distance.csv"))
decay.to_csv(os.path.join(OUT, "sequence_decay.csv"), index=False)
or_rec.to_csv(os.path.join(OUT, "sequence_or_given_reception.csv"))
print("saved")

saved
